# Experimento  — Adição do Nitrogênio (5 Variáveis) com 3 Classificações e Sem Balanceamento

Após a realização dos experimentos iniciais utilizando apenas quatro variáveis, decidiu-se adicionar a variável de nitrogênio ao modelo para analisar como esse novo parâmetro impactaria o comportamento do Random Forest.

Além disso, como os experimentos anteriores mostraram dificuldades principalmente na identificação da classe `Crítica`, a inclusão do nitrogênio pode ajudar o modelo a encontrar padrões mais claros relacionados a cenários de degradação ambiental.

Neste experimento, serão utilizadas:
- 5 variáveis de entrada;
- 3 classificações (`Adequada`, `Atenção` e `Crítica`);
- sem aplicação de balanceamento.


O objetivo deste experimento é observar:
- se o nitrogênio melhora a capacidade de generalização do modelo;
- se há impacto na redução do overfitting;
- se o recall da classe `Crítica` melhora;
- se o modelo consegue aprender padrões ambientais mais relevantes;
- como a adição de uma nova variável influencia as métricas gerais do Random Forest.

Além disso, os resultados deste experimento serão comparados com:
1. o experimento utilizando 4 variáveis e 3 classificações sem balanceamento;
2. o experimento antigo utilizando 5 variáveis e 4 classificações sem balanceamento.

Essa comparação é importante para entender:
- o impacto da redução de classes;
- o impacto da adição do nitrogênio;
- como o modelo responde a diferentes distribuições de classes;
- quais configurações apresentam melhor equilíbrio entre generalização e capacidade preditiva.

## Preparação do ambiente


In [ ]:
# IMPORTAÇÃO DE BIBLIOTECAS
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

SEED = 42

In [ ]:
# DETECÇÃO DE AMBIENTE
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# CONFIGURAÇÃO DE CAMINHO E CARREGAMENTO
if IN_COLAB:
    print("Ambiente Google Colab detectado.")

    drive.mount('/content/drive')

    DATA_PATH = Path(
        "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/amostra_rotulada_3.parquet"
    )

else:
    print("Ambiente local/VS Code detectado.")

    DATA_PATH = Path("../../dataset/amostra_rotulada_3.parquet")

# LEITURA DO DATASET
df = pd.read_parquet(DATA_PATH)

print("Dataset Parquet carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")

df.head()

Ambiente Google Colab detectado.
Mounted at /content/drive
Dataset Parquet carregado com sucesso.
Shape do dataset: (141399, 22)


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),...,CCME_Values,CCME_WQI,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status
0,Canada,CZPOH_1117,River,05-03-2012,0.092790,2.25455,9.43636,0.06100,7.50000,9.40000,...,93.116725,Good,1,1,1,1,3.7,1,5,Adequada
1,Canada,FISW_32,Lake,02-12-2003,0.043792,2.13333,9.82400,0.00200,7.79000,12.00000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Adequada
2,Canada,CZPOH_1036,River,12-03-2018,0.085920,2.01667,11.78333,0.02568,7.36667,8.91667,...,100.000000,Excellent,1,1,1,1,3.7,1,5,Adequada
3,Canada,IEEA_10_32,Lake,08-06-2001,0.015920,0.55000,9.82400,0.00400,7.79000,16.80000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Adequada
4,Canada,ES0910524,River,11-09-2012,0.150000,2.13333,10.32500,0.07250,7.79000,8.32500,...,92.882835,Good,1,1,1,1,2.0,1,5,Adequada


In [ ]:
# preparando o ambiente para machine learning
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

In [ ]:
# definição de x e y
X = df[
    [
        "Temperature (cel)",
        "Orthophosphate (mg/l)",
        "Country",
        "Waterbody Type",
        "Nitrogen (mg/l)"
    ]
]

y = df["conama_status"]

In [ ]:
#train_test split

SEED = 42

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)


Treino: (113119, 5)
Teste: (28280, 5)


In [ ]:
# pré-processamento

categorical_features = [
    "Country",
    "Waterbody Type"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [ ]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),

        (
            "classifier",
            RandomForestClassifier(
                random_state=SEED

            )
        )
    ]
)

In [ ]:
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Country',
                                                   'Waterbody Type'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

## Avaliação das Métricas de Treino

Antes de analisar os resultados do conjunto de teste, também foi realizada a avaliação das métricas no conjunto de treino.

Essa etapa é importante porque permite comparar o comportamento do modelo entre treino e teste, ajudando na identificação de possíveis sinais de overfitting.

Por esse motivo, não basta apenas observar uma alta acurácia no treino. O ideal é que os resultados de treino e teste permaneçam relativamente próximos, indicando que o modelo conseguiu aprender padrões mais generalizáveis ao invés de apenas memorizar os dados.

Além da acurácia, também foram analisadas métricas como precision, recall e F1-score, permitindo observar como o modelo se comporta em cada classe individualmente.

In [ ]:
y_train_pred = model.predict(X_train)

In [ ]:
train_accuracy = accuracy_score(y_train, y_train_pred)

train_precision = precision_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_recall = recall_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_f1 = f1_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_confusion_matrix = confusion_matrix(
    y_train,
    y_train_pred
)

print("Train Accuracy:")
print(train_accuracy)

print("Train Precision:")
print(train_precision)

print("Train Recall:")
print(train_recall)

print("Train F1:")
print(train_f1)

print("Train Confusion Matrix:")
print(train_confusion_matrix)

Train Accuracy:
0.9446158470283507
Train Precision:
0.9441780697970729
Train Recall:
0.9446158470283507
Train F1:
0.9442030625318345
Train Confusion Matrix:
[[80385  2390     0]
 [ 3734 25501     3]
 [   28   110   968]]


In [ ]:
y_pred = model.predict(X_test)

In [ ]:
# Teste com 5 variáveis - sem balanceamento
print("Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.803995756718529

Classification Report:
              precision    recall  f1-score   support

    Adequada       0.85      0.90      0.88     20694
     Atenção       0.64      0.57      0.60      7309
     Crítica       0.20      0.05      0.08       277

    accuracy                           0.80     28280
   macro avg       0.57      0.51      0.52     28280
weighted avg       0.79      0.80      0.80     28280


Confusion Matrix:
[[18567  2110    17]
 [ 3115  4156    38]
 [   57   206    14]]



# Resultados Obtidos — Experimento com 5 Variáveis, 3 Classificações e Sem Balanceamento

Os resultados deste experimento mostraram um avanço importante em relação aos experimentos anteriores.

A principal diferença deste teste foi:
- adição da variável de nitrogênio;
- utilização de apenas 3 classificações (`Adequada`, `Atenção` e `Crítica`);
- ausência de balanceamento.

---

# Comparação com o Experimento de 4 Variáveis e 3 Classificações

Quando comparado ao experimento utilizando apenas 4 variáveis e 3 classificações sem balanceamento, foi possível observar uma melhora significativa no desempenho geral do modelo.

## Accuracy de treino

### 4 variáveis:
```text
0.900
```

### 5 variáveis:

```text
0.945
```

A accuracy de treino aumentou consideravelmente após a adição do nitrogênio.

Isso indica que o modelo conseguiu aprender padrões mais complexos utilizando a nova variável.

---

## Accuracy de teste

### 4 variáveis:

```text
0.744
```

### 5 variáveis:

```text
0.804
```

A accuracy de teste também aumentou bastante.

Esse resultado mostra que o nitrogênio realmente adicionou informação relevante ao modelo e melhorou sua capacidade de generalização.

---

# Impacto nas Classes

## Classe `Adequada`

### Antes:

* Precision: 0.81
* Recall: 0.87
* F1-score: 0.84

### Depois:

* Precision: 0.85
* Recall: 0.90
* F1-score: 0.88

A classe `Adequada` apresentou melhora em todas as métricas.

Isso mostra que o modelo passou a identificar melhor padrões associados a águas em condições adequadas.

---

## Classe `Atenção`

### Antes:

* Precision: 0.51
* Recall: 0.42
* F1-score: 0.46

### Depois:

* Precision: 0.64
* Recall: 0.57
* F1-score: 0.60

Essa foi uma das melhorias mais importantes do experimento.

O modelo passou a identificar muito melhor situações intermediárias de qualidade da água.

Isso é extremamente relevante ambientalmente, porque a classe `Atenção` representa cenários próximos de degradação ambiental.

---

## Classe `Crítica`

### Antes:

* Precision: 0.08
* Recall: 0.03
* F1-score: 0.04

### Depois:

* Precision: 0.20
* Recall: 0.05
* F1-score: 0.08

Mesmo ainda apresentando valores baixos, a classe `Crítica` também apresentou melhora.

O recall praticamente dobrou:

```text
0.03 → 0.05
```

A precision também aumentou bastante:

```text
0.08 → 0.20
```

Isso indica que o nitrogênio ajudou o modelo a reconhecer melhor padrões relacionados a cenários críticos.

---

# Comparação com o Experimento de 5 Variáveis e 4 Classificações

Também foi realizada comparação com o experimento antigo utilizando:

* 5 variáveis;
* 4 classificações;
* sem balanceamento.

---

## Accuracy de teste

### 4 classificações:

```text
0.765
```

### 3 classificações:

```text
0.804
```

A redução para 3 classificações melhorou significativamente a capacidade preditiva do modelo.

---

# Por que isso aconteceu?

Quando existiam 4 classificações:

* os dados ficavam mais fragmentados;
* algumas classes possuíam poucas amostras;
* o modelo precisava aprender fronteiras mais específicas entre categorias.

Ao reduzir para 3 classificações:

* houve maior concentração de amostras;
* a distribuição ficou menos fragmentada;
* o modelo conseguiu aprender padrões mais consistentes.

Além disso:

* a antiga classe `Boa` foi absorvida pela classe `Atenção`;
* isso aumentou bastante a quantidade de exemplos disponíveis nessa categoria;
* consequentemente, o modelo passou a identificar melhor padrões intermediários.

---

# Overfitting

## 4 variáveis e 3 classes

Diferença treino/teste:

```text
0.900 - 0.744 = 0.156
```

## 5 variáveis e 3 classes

Diferença treino/teste:

```text
0.945 - 0.804 = 0.141
```

Mesmo com aumento da accuracy de treino, o overfitting diminuiu levemente.

Isso é um resultado muito importante.

Significa que:

* o modelo não apenas decorou mais;
* ele também conseguiu generalizar melhor.

---

# Interpretação Geral

Os resultados indicam que:

* a variável de nitrogênio contribuiu positivamente para o modelo;
* trabalhar com 3 classificações foi mais eficiente do que trabalhar com 4;
* o modelo conseguiu melhorar tanto accuracy quanto métricas das classes;
* houve melhora principalmente na classe `Atenção`;
* a classe `Crítica` continua sendo o maior desafio devido à baixa quantidade de amostras.

Mesmo assim, o experimento mostrou evolução importante na capacidade do modelo de aprender padrões ambientais relevantes.




In [ ]:
import pandas as pd

# =========================
# DADOS DOS EXPERIMENTOS
# =========================

dados = {
    "Métrica": [
        "Accuracy Treino",
        "Accuracy Teste",
        "Overfitting",

        "Precision Adequada/Excelente",
        "Recall Adequada/Excelente",
        "F1 Adequada/Excelente",

        "Precision Atenção",
        "Recall Atenção",
        "F1 Atenção",

        "Precision Boa",
        "Recall Boa",
        "F1 Boa",

        "Precision Crítica",
        "Recall Crítica",
        "F1 Crítica",
    ],

    "5 Variáveis | 3 Classes": [
        0.945,
        0.804,
        0.141,

        0.85,
        0.90,
        0.88,

        0.64,
        0.57,
        0.60,

        "-",
        "-",
        "-",

        0.20,
        0.05,
        0.08,
    ],

    "5 Variáveis | 4 Classes": [
        0.932,
        0.765,
        0.167,

        0.84,
        0.91,
        0.87,

        0.41,
        0.24,
        0.30,

        0.50,
        0.42,
        0.46,

        0.16,
        0.06,
        0.09,
    ],

    "4 Variáveis | 3 Classes": [
        0.900,
        0.744,
        0.156,

        0.81,
        0.87,
        0.84,

        0.51,
        0.42,
        0.46,

        "-",
        "-",
        "-",

        0.08,
        0.03,
        0.04,
    ]
}

# =========================
# DATAFRAME
# =========================

df = pd.DataFrame(dados)

# =========================
# ESTILIZAÇÃO
# =========================

styled_df = (
    df.style
    .set_properties(**{
        'text-align': 'center',
        'font-size': '14px'
    })
    .set_table_styles([
        {
            'selector': 'th',
            'props': [
                ('background-color', '#0f766e'),
                ('color', 'white'),
                ('font-weight', 'bold'),
                ('text-align', 'center'),
                ('padding', '12px'),
                ('border', '1px solid #d1d5db')
            ]
        },
        {
            'selector': 'td',
            'props': [
                ('padding', '10px'),
                ('border', '1px solid #e5e7eb')
            ]
        },
        {
            'selector': 'tr:nth-child(even)',
            'props': [
                ('background-color', '#ecfeff')
            ]
        },
        {
            'selector': 'tr:nth-child(odd)',
            'props': [
                ('background-color', 'white')
            ]
        }
    ])
    .hide(axis="index")
)

# =========================
# EXIBIÇÃO
# =========================

display(styled_df)

Métrica,5 Variáveis | 3 Classes,5 Variáveis | 4 Classes,4 Variáveis | 3 Classes
Accuracy Treino,0.945000,0.932000,0.900000
Accuracy Teste,0.804000,0.765000,0.744000
Overfitting,0.141000,0.167000,0.156000
Precision Adequada/Excelente,0.850000,0.840000,0.810000
Recall Adequada/Excelente,0.900000,0.910000,0.870000
F1 Adequada/Excelente,0.880000,0.870000,0.840000
Precision Atenção,0.640000,0.410000,0.510000
Recall Atenção,0.570000,0.240000,0.420000
F1 Atenção,0.600000,0.300000,0.460000
Precision Boa,-,0.500000,-
